# Day 1 · Hands-on 3 — p-values and what they don't tell you
**About 20 min · work in pairs** (one person drives the keyboard, the other reads, predicts and questions — swap halfway)


Patients in GBSG2 either received hormonal therapy (`horTh` = "yes") or not. Before looking at outcomes, a careful analyst checks whether the two groups were **comparable at baseline**. We'll use this check to practise reading p-values.



> **Two AI roles — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to write, change, fix or explain *code*, including new simulations.
> - 🎓 **Claude / ChatGPT (other tab) = tutor.** Use it to explain ideas, quiz you and critique your own work. Ask for its reasoning, and check one claim yourself in the notebook.


**How this notebook works:** run cells from top to bottom. Each code cell has exactly one line marked `# 🔧 CHANGE THIS` — that is the only thing you need to edit. Lines starting with 📋 are results to enter in the shared sheet.

In [ ]:
# ▶️ Run this cell first (about 20 seconds). Nothing to change here.
%pip install -q lifelines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from lifelines.datasets import load_gbsg2

df = load_gbsg2()
print(f"✅ Loaded GBSG2: {len(df)} patients, {df.shape[1]} variables")

## A. Were the groups comparable?
### 🧭 Concept
A p-value answers: *if there were truly no difference between the groups, how surprising would a difference at least this large be?* It is **not** the probability that the null hypothesis is true.

✍️ **Predict first:** for `age`, will the p-value be large or small? Then try the other variables: `tsize`, `pnodes`, `progrec`, `estrec`, `menostat`, `tgrade`.

In [ ]:
# 🔧 CHANGE THIS: baseline variable to compare between therapy groups
variable = "age"

yes, no = df[df.horTh == "yes"], df[df.horTh == "no"]

if df[variable].dtype.kind in "if":                      # numeric variable -> Welch t-test
    a, b = yes[variable], no[variable]
    diff = a.mean() - b.mean()
    va, vb = a.var(ddof=1) / len(a), b.var(ddof=1) / len(b)
    se = np.sqrt(va + vb)
    dof = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    t_crit = stats.t.ppf(0.975, dof)
    p = stats.ttest_ind(a, b, equal_var=False).pvalue
    print(f"Mean {variable}: therapy = {a.mean():.1f}, no therapy = {b.mean():.1f}")
    print(f"Difference: {diff:.1f}   95% CI [{diff - t_crit * se:.1f}, {diff + t_crit * se:.1f}]")
    print(f"Welch t-test p-value: {p:.2g}")
    df.boxplot(column=variable, by="horTh", figsize=(5, 3.5)); plt.suptitle(""); plt.show()
else:                                                     # categorical variable -> chi-square test
    table = pd.crosstab(df.horTh, df[variable])
    p = stats.chi2_contingency(table)[1]
    print("Percentage of each group in each category:")
    print((pd.crosstab(df.horTh, df[variable], normalize="index") * 100).round(1))
    print(f"\nChi-square test p-value: {p:.2g}")

### ✅ Concept check
Age gives a very small p-value and `menostat` too. Which of these conclusions follow?
1. The groups differ in age by more than chance alone would easily produce.
2. Hormonal therapy causes patients to be older.
3. Comparing outcomes between these groups could be misleading.
4. There is a less than 0.1% probability that the groups have the same mean age.

<details><summary>Show answers</summary>

**1** and **3**. **2** reverses the logic: age was there before treatment, so the likely story is that *who received therapy* depended on age or menopausal status. **4** is the most common p-value misreading. The p-value is calculated *assuming* equal means; it cannot also be the probability of that assumption.

Keep **3** in mind: when groups differ at baseline, a difference in outcomes may come from these baseline differences rather than from the treatment. This is **confounding**, which we tackle on Day 3.
</details>

## B. Significant is not the same as important
### 🧭 Concept
Imagine a drug that shrinks tumours by **1 mm** on average. That is clinically trivial. Here is a simulated trial with that effect. Try 50, 500, 5000 and 50000 patients per group and watch the p-value and the confidence interval.

In [ ]:
# 🔧 CHANGE THIS: patients per group
n_per_group = 50

true_effect = 1.0                     # mm — clinically trivial
sd = df["tsize"].std()                # realistic variability, from GBSG2
rng = np.random.default_rng()
control = rng.normal(30, sd, n_per_group)
treated = rng.normal(30 - true_effect, sd, n_per_group)

diff = control.mean() - treated.mean()
se = np.sqrt(control.var(ddof=1) / n_per_group + treated.var(ddof=1) / n_per_group)
p = stats.ttest_ind(control, treated, equal_var=False).pvalue
print(f"{n_per_group} per group: estimated effect = {diff:.2f} mm, "
      f"95% CI [{diff - 1.96 * se:.2f}, {diff + 1.96 * se:.2f}], p = {p:.2g}")

## 🤖 Go deeper with AI
Three ways to keep learning about p-values: now, if you finish early, or after the workshop.

**🎓 Explain and quiz me** (Claude / ChatGPT):
> *Explain what a p-value is to a cell biologist using a non-statistical analogy. Then ask me 3 questions, one at a time, and tell me where my answers go wrong.*

**🧪 See it in a simulation** (🧮 Gemini in Colab, in a new cell at the end of this notebook):
> *Simulate 10,000 two-group experiments with no true difference and plot the histogram of p-values. Then repeat with a real difference of half an SD. Explain why the two histograms look different.*

**🔬 Apply it to my work** (Claude / ChatGPT; replace the brackets with your own details):
> *Here is a results sentence from a paper I'm reading: [paste]. Rewrite it so that it says only what the p-value and the effect estimate support.*

✔ **Check one claim yourself:** With no true difference the histogram should be flat: about 5% of p-values below 0.05.

## 💬 Discuss with your partner
- In a perfectly **randomized** trial, any baseline difference is due to chance by design. What would a baseline p-value be testing then, and why do reporting guidelines (CONSORT) advise against such tests?
- Here the groups clearly differ. What does that suggest about how hormonal therapy was assigned in this dataset?
- In section B, what would you report instead of "p < 0.05"?